# Analiza dialogowa scenariusza: synteza i prezentacja

## Wersja studencka — moduł 4: SYNTEZA

W poprzednich modułach zebrałeś trzy warstwy analizy:
- **Sieć współwystąpień** — kto z kim jest w scenach (`graf_postaci.graphml`)
- **Głos i płeć** — kto mówi i ile (`dialogi.csv`, `postacie.csv`)
- **Emocje** — z jakim ładunkiem emocjonalnym (`emocje.csv`, `emocje_postaci.csv`)

Ten notatnik łączy wszystkie trzy warstwy w jedno wyjście gotowe do prezentacji. Efektem będzie `bundle.json`, wzbogacony graf i archiwum ZIP — a na ich podstawie wygenerujesz prezentację HTML za pomocą gotowego promptu.

## Jak pracować z tym notatnikiem

1. Upewnij się, że masz wszystkie pliki z poprzednich modułów: `graf_postaci.graphml`, `dialogi.csv`, `postacie.csv`, `emocje.csv`, `emocje_postaci.csv`.
2. Skopiuj prompt z bieżącego kroku do modelu AI, wklej kod, uruchom i porównaj z sekcją **Po uruchomieniu powinieneś zobaczyć**.
3. **Etapy 3 i 4** mają prawie gotowy prompt — wypełniasz jedną lukę i uruchamiasz komórkę.
4. Etap 5 to krok wykonywany poza notatnikiem (w oknie czatu z modelem AI).

## Wypełnij jedną lukę — tak jak w NB02

W NB1 czytałeś gotowe prompty. W NB2 uzupełniałeś jedną lukę. Tutaj jest tak samo.

**Jak to działa:**
Dwa kroki (3A i 4A) mają komórkę kodu z prawie gotowym `moj_prompt`. Znajdź w nim `[UZUPEŁNIJ: …]` lub `[WPISZ: …]`, wpisz swoją odpowiedź, uruchom komórkę — asercja sprawdzi, czy luka jest wypełniona. Jeśli przejdzie, skopiuj wypisany prompt do asystenta AI.

To może być proste — chodzi o to, żebyś podjął jedną rzeczywistą decyzję o SWOIM filmie.

---
## Etap 1/5 — Wczytanie i inspekcja danych

Zaczynamy od załadowania wszystkich plików i sprawdzenia, że wszystko jest na miejscu.

In [ ]:
# === ETAP 1/5 · WCZYTANIE DANYCH ===
import os
wymagane = ["graf_postaci.graphml", "dialogi.csv", "postacie.csv",
            "emocje.csv", "emocje_postaci.csv"]
brak = [f for f in wymagane if not os.path.exists(f)]
if brak:
    raise FileNotFoundError(
        f"⛔ Brakuje plików: {brak}\n"
        "Pobierz je z poprzednich modułów i wgraj do tego środowiska."
    )
print("📍 Etap 1/5 · Wczytanie danych. Wszystkie 5 plików znalezione ✓")

### Krok 1A. Wczytanie plików i raport o kolumnach

#### Cel i sens analityczny

Zanim scalimy dane z trzech modułów, musimy wiedzieć, jakie kolumny mamy w każdym pliku. Dane z różnych źródeł rzadko pasują do siebie „od razu".

#### Prompt dla modelu

```text
Kontekst:
Masz pięć plików z poprzednich modułów analizy scenariusza filmowego.

Wejście:
Pliki: `graf_postaci.graphml`, `dialogi.csv`, `postacie.csv`, `emocje.csv`, `emocje_postaci.csv`.

Zadanie:
Wczytaj wszystkie pięć plików. Dla każdego pliku CSV wypisz: liczbę wierszy i dokładne nazwy kolumn. Dla pliku GraphML wypisz: liczbę węzłów i krawędzi. Wczytaj graf do zmiennej o nazwie `G` (networkx). Wczytaj CSV-e do zmiennych o nazwach `dialogi`, `postacie`, `emocje`, `emocje_postaci`. Sprawdź, czy kolumna z nazwą postaci istnieje we wszystkich czterech CSV-ach i pokaż kilka przykładów nazw postaci z każdego pliku.

Pokaż wynik:
- tabelkę: plik | liczba wierszy/węzłów | kolumny,
- porównanie 5 przykładowych nazw postaci z `postacie` i `emocje_postaci`.

Warunek poprawności:
Nazwy postaci powinny wyglądać tak samo w obu plikach. Jeśli się różnią, zaznacz to.

Jeśli wystąpi błąd:
Wyświetl, którego pliku nie udało się wczytać i podaj możliwą przyczynę.

Nie rób jeszcze:
Nie scalaj danych ani nie modyfikuj grafu.
```

In [ ]:
# tu wklej kod wygenerowany przez model

In [ ]:
# NORMALIZACJA — ujednolicenie nazwy kolumny postaci we wszystkich tabelach
# Uruchom zaraz po komórce z wczytaniem plików. Działa automatycznie.

_SZUKAJ = ('postac', 'postać', 'character', 'speaker', 'bohater',
           'imie', 'imię', 'name', 'nazwa', 'person', 'char')
_KANON  = 'postać'

_tabele = {
    'dialogi':        dialogi,
    'postacie':       postacie,
    'emocje':         emocje,
    'emocje_postaci': emocje_postaci,
}

print("🔍 Wykrywanie i normalizacja kolumny postaci:\n")
_problem = False
for nazwa, df in _tabele.items():
    kol = next((c for c in df.columns
                if any(k in c.lower() for k in _SZUKAJ)), None)
    if kol is None:
        print(f"  ❌ {nazwa}: nie znaleziono — dostępne kolumny: {list(df.columns)}")
        _problem = True
    elif kol != _KANON:
        df.rename(columns={kol: _KANON}, inplace=True)
        print(f"  🔄 {nazwa}: '{kol}' → '{_KANON}'")
    else:
        print(f"  ✅ {nazwa}: '{_KANON}' — OK")

if _problem:
    raise ValueError(
        "\n⛔ Nie udało się automatycznie znaleźć kolumny postaci w jednym z plików.\n"
        "Sprawdź listę kolumn wypisaną powyżej i dodaj poniżej ręczne przemianowanie:\n\n"
        "    dialogi.rename(columns={'TWOJA_NAZWA': 'postać'}, inplace=True)\n\n"
        "Następnie uruchom tę komórkę ponownie."
    )
print(f"\n✅ Wszystkie tabele mają kolumnę '{_KANON}'. Przejdź do Etapu 2.")

#### Po uruchomieniu powinieneś zobaczyć

- Listę plików z liczbą wierszy i nazwami kolumn.
- Potwierdzenie, że graf ma węzły i krawędzie.
- Porównanie nazw postaci między plikami.

> **Jeśli nazwy postaci się różnią** (spacje, wielkość liter) — poproś model o krok normalizujący przed przejściem dalej.

---
## Etap 2/5 — Wzbogacenie sieci i pakiet danych

To główny krok tego modułu. Dołączamy do każdego węzła sieci dane z poprzednich modułów i obliczamy agregaty potrzebne do wykresów w prezentacji.

In [ ]:
# === ETAP 2/5 · WZBOGACENIE SIECI ===
assert 'G' in dir(), "⛔ Najpierw uruchom Etap 1 — brak grafu `G`."
assert 'postacie' in dir() and 'emocje' in dir(), \
    "⛔ Najpierw uruchom Etap 1 — brak tabel."
import networkx as nx
print(f"📍 Etap 2/5 · Wzbogacenie sieci. "
      f"Graf: {G.number_of_nodes()} węzłów, {G.number_of_edges()} krawędzi.")

### Krok 2A. Wzbogacenie grafu i zapis bundle.json

#### Cel i sens analityczny

Wzbogacony graf łączy wszystko: wiesz nie tylko, kto z kim rozmawia, ale czyim głosem dominuje scena, jaką emocją jest nacechowany i jakiej jest płci. `bundle.json` zbiera te dane w formacie gotowym do wklejenia do modelu generującego prezentację.

#### Prompt dla modelu

```text
Kontekst:
Masz wczytane: graf sieci w zmiennej `G` (networkx), tabele `postacie`, `emocje_postaci` i `emocje`. Nazwy kolumn mogą się różnić od podanych — użyj tych, które faktycznie są w plikach.

Wejście:
Graf `G` i cztery tabele wczytane w Etapie 1.

Zadanie:
Wykonaj trzy rzeczy:

1. WZBOGAĆ GRAF: dla każdego węzła grafu dołącz atrybuty z `postacie` i `emocje_postaci`: płeć (gender), liczba słów (word_count), dominująca emocja (dominant_emotion), średnia walencja (avg_valence). Węzły bez danych: gender="?", word_count=0, dominant_emotion="neutralna", avg_valence=0.0. Zapisz wzbogacony graf do `graf_postaci_wzbogacony.graphml`.

2. OBLICZ AGREGATY:
   a. top_speakers: top 15 postaci wg liczby słów — z płcią i procentem łącznej mowy.
   b. speech_by_gender: łączna mowa wg K/M/? — i ich procent.
   c. emotion_arc: dla każdej sceny: średnia walencja, wygładzona walencją (okno 5). Lista par (numer_sceny, rolling_valence).
   d. emotion_by_gender: dla K i M — ile % kwestii przypada na każdą z 6 emocji.

3. ZAPISZ bundle.json (UTF-8, indent=2):
{
  "top_speakers": [{"name":..., "gender":..., "word_count":..., "pct_of_total":...}],
  "speech_by_gender": {"K":{"total_words":..., "pct":...}, "M":{...}, "?":{...}},
  "emotion_arc": [{"scene":..., "avg_valence":..., "rolling_valence":...}],
  "emotion_by_gender": {
    "K": {"radość":..., "smutek":..., "złość":..., "strach":..., "zaskoczenie":..., "neutralna":...},
    "M": {tak samo}
  },
  "network": {
    "nodes": [{"id":..., "gender":..., "word_count":..., "scene_count":..., "dominant_emotion":..., "avg_valence":...}],
    "edges": [{"source":..., "target":..., "weight":...}]
  }
}
Zachowaj dane bundle jako zmienną o nazwie `bundle`.

Pokaż wynik:
- potwierdzenie zapisu obu plików z rozmiarami,
- 5 przykładowych węzłów z wzbogaconego grafu (postać + atrybuty),
- pierwsze i ostatnie 3 wpisy łuku emocjonalnego.

Warunek poprawności:
Suma % speech_by_gender = 100%. Procenty emocji dla K sumują się do 100%, tak samo dla M.

Jeśli wystąpi błąd:
Wypisz, na którym kroku (wzbogacanie / agregaty / bundle) wystąpił problem.
```

In [ ]:
# tu wklej kod wygenerowany przez model

#### Po uruchomieniu powinieneś zobaczyć

- Potwierdzenie zapisu `graf_postaci_wzbogacony.graphml` i `bundle.json`.
- Kilka węzłów z atrybutami: płeć, liczba słów, emocja, walencja.
- Pierwsze i ostatnie wpisy łuku emocjonalnego.

---
## Etap 3/5 — Podgląd sieci wzbogaconej

Czwarty wykres-bohater całego modułu: ta sama sieć co w module 1, ale teraz kolor węzła = płeć i rozmiar = mowa. Pokazuje jednym obrazem, kto ma głos w tej sieci.

In [ ]:
# === ETAP 3/5 · SIEĆ WZBOGACONA ===
assert 'G' in dir(), "⛔ Najpierw uruchom Etapy 1 i 2 — brak grafu."
assert 'bundle' in dir() or 'postacie' in dir(), \
    "⛔ Najpierw uruchom Etap 2 — brak danych postaci."
import networkx as nx
print(f"📍 Etap 3/5 · Wizualizacja sieci wzbogaconej. "
      f"Graf: {G.number_of_nodes()} węzłów.")

In [ ]:
# HIPOTEZA — wpisz ZANIM narysujesz sieć
# Czy postać z największą liczbą połączeń w sieci (centralnie osadzona)
# to też ta, która mówi najwięcej (największy węzeł)?
# A może te dwa wymiary się rozchodzą?
moja_hipoteza_3 = ""   # np. "tak — VINCENT jest i centralny, i dużo mówi"
assert moja_hipoteza_3.strip(), \
    "⛔ Najpierw postaw hipotezę o relacji centralności i mowy."
print("✏️  Hipoteza zapisana. Teraz napisz prompt i wygeneruj sieć.")

### Krok 3A. Wykres sieci z atrybutami głosu i płci

#### Cel i sens analityczny

Wizualizacja wzbogaconej sieci to pomost między analizą sieciową a dialogową. Wcześniej widziałeś, kto z kim *bywa*. Teraz widzisz, kto z kim *bywa i ile mówi*. Postać centralnie połączona, ale małych rozmiarów — to ktoś, kto dużo *obserwuje*, ale rzadko *mówi*.

In [ ]:
# PRZEGLĄD — uruchom przed napisaniem promptu
assert 'G' in dir(), "⛔ Najpierw uruchom Etapy 1 i 2."
import networkx as nx
print(f"📋 Graf `G`: {G.number_of_nodes()} węzłów, {G.number_of_edges()} krawędzi\n")
print("── Atrybuty przykładowych węzłów (pierwsze 5) ─────────────────")
for node, attrs in list(G.nodes(data=True))[:5]:
    print(f"  {node}: {attrs}")
print("\n── Atrybuty przykładowych krawędzi (pierwsze 5) ───────────────")
for u, v, attrs in list(G.edges(data=True))[:5]:
    print(f"  {u} — {v}: {attrs}")

#### Wypełnij jedną lukę

Prompt jest prawie gotowy. Znajdź w nim `[UZUPEŁNIJ: …]` i uzupełnij — to decyzja o tym, ile postaci będzie miało etykiety na wykresie sieci.

**Checklista — co sprawdzić po uruchomieniu wygenerowanego kodu:**
- [ ] Węzły mają różne rozmiary (większy = więcej słów)
- [ ] Kolory różne dla K, M i ? (trzy różne odcienie)
- [ ] Etykiety tylko przy wybranych postaciach, nie wszystkich
- [ ] Plik `siec_wzbogacona.png` zapisany

In [ ]:
# TWOJA KOLEJ: uzupełnij jedną lukę [UZUPEŁNIJ: …], potem uruchom.
moj_prompt = """
Kontekst:
Masz wzbogacony graf sieci postaci w zmiennej `G` (networkx). Węzły mają atrybuty: `gender` (K/M/?), `word_count` (liczba słów), `dominant_emotion`, `avg_valence`. Krawędzie mają atrybut `weight` (liczba wspólnych scen).

Wejście:
Zmienna `G` z Etapu 2. Biblioteki `networkx` i `matplotlib` są już zainstalowane.

Zadanie:
1. Pobierz `word_count` dla każdego węzła i przeskaluj na rozmiary punktów: minimum 300, maksimum 3000 (dla węzłów z word_count=0 użyj 100).
2. Przypisz kolory wg `gender`: K = '#e07b8a', M = '#5b8db8', ? = '#aaaaaa'.
3. Ustal grubość krawędzi proporcjonalnie do `weight`: minimum 0.3, maksimum 3.0.
4. Wybierz [UZUPEŁNIJ: ile postaci z największym word_count ma mieć etykiety? — typowo 8–15, zależy od liczby węzłów w grafie] węzłów — tylko dla nich wyświetl etykiety z imionami.
5. Narysuj sieć z układem spring_layout (seed=42). Dodaj tytuł "Sieć postaci — głos i płeć" i legendę kolorów płci.
6. Zapisz jako `siec_wzbogacona.png` (dpi=150).

Pokaż wynik:
Wyświetl rysunek w notatniku i potwierdź zapis pliku z jego rozmiarem.

Warunek poprawności:
Węzły mają różne rozmiary (nie wszystkie jednakowe). Plik `siec_wzbogacona.png` istnieje na dysku.

Jeśli wystąpi błąd:
Wypisz nazwę atrybutu, który sprawia problem. Sprawdź, czy `list(G.nodes(data=True))[:3]` zawiera oczekiwane klucze.

Nie rób jeszcze:
Nie modyfikuj grafu ani nie zapisuj żadnych innych plików.
"""
assert "[UZUPEŁNIJ" not in moj_prompt, \
    "⛔ Pozostała luka [UZUPEŁNIJ: …] — wpisz liczbę postaci z etykietami."
assert "siec_wzbogacona.png" in moj_prompt, \
    "⛔ Prompt musi zawierać nazwę pliku wynikowego: siec_wzbogacona.png."
print("✅ Prompt gotowy. Skopiuj poniższą treść do asystenta AI:\n")
print(moj_prompt)

In [ ]:
# tu wklej kod wygenerowany przez model

#### Po uruchomieniu powinieneś zobaczyć

- Graf sieci z węzłami o zróżnicowanych rozmiarach i kolorach.
- Etykiety 12 postaci z największą liczbą słów.
- Legendę kolorów płci i potwierdzenie zapisu `siec_wzbogacona.png`.

In [ ]:
# CHECKPOINT — uzupełnij na podstawie wygenerowanego wykresu
wezel_centralny = ""  # postać wyglądająca na najbardziej centralną (dużo połączeń)
wezel_mowny    = ""  # postać z największym węzłem (dużo słów)
assert wezel_centralny.strip() and wezel_mowny.strip(), \
    "⛔ Przyjrzyj się sieci i wpisz nazwy obu postaci."
print(f"📊 Centralnie osadzona: '{wezel_centralny}' | Najwięcej mówi: '{wezel_mowny}'")
if wezel_centralny.strip().upper() == wezel_mowny.strip().upper():
    print("➡️  To ta sama postać — centralność i mowa się pokrywają.")
else:
    print("➡️  To różne postacie — masz tu ciekawy materiał interpretacyjny!")
h = moja_hipoteza_3.lower()
if wezel_centralny.strip().lower() in h or wezel_mowny.strip().lower() in h:
    print("✅ Hipoteza trafna przynajmniej częściowo.")
else:
    print("🤔 Dane zaskoczyły — zastanów się, dlaczego!")

#### Pytanie interpretacyjne

Czy postacie centralnie osadzone w sieci (wiele połączeń) to też te, które najwięcej mówią (duże węzły)? Albo inaczej: czy są postacie *drobne* (mało mówią), a jednak *centralnie* osadzone? Co to może znaczyć narracyjnie?

---
## Etap 4/5 — Archiwum do pobrania

Zbieramy wszystkie pliki wyjściowe i tworzymy archiwum ZIP gotowe do pobrania z Colab.

In [ ]:
# === ETAP 4/5 · ARCHIWUM ZIP ===
assert os.path.exists("bundle.json"), \
    "⛔ Najpierw uruchom Etap 2 — brak pliku bundle.json."
print("📍 Etap 4/5 · Tworzenie archiwum ZIP.")

In [ ]:
# PRZEGLĄD — uruchom przed napisaniem promptu
import os
wymagane = ["bundle.json", "graf_postaci_wzbogacony.graphml",
            "siec_wzbogacona.png", "dialogi.csv", "postacie.csv",
            "emocje.csv", "emocje_postaci.csv"]
print("📋 Status plików do spakowania:\n")
for f in wymagane:
    if os.path.exists(f):
        print(f"  ✅ {f} ({os.path.getsize(f)/1024:.1f} KB)")
    else:
        print(f"  ❌ {f} — BRAK")

### Krok 4A. Zapis archiwum ZIP

#### Cel i sens analityczny

Prezentację HTML wygenerujesz poza notatnikiem — potrzebujesz wszystkich plików lokalnie. ZIP jest wygodniejszy niż pobieranie po jednym.

#### Wypełnij jedną lukę

Prompt jest prawie gotowy. Znajdź `[WPISZ: …]` i wpisz tytuł swojego filmu — stanie się nazwą archiwum ZIP, które pobierzesz.

**Checklista — co sprawdzić po uruchomieniu wygenerowanego kodu:**
- [ ] Archiwum ZIP dostępne do pobrania (link lub plik)
- [ ] Lista spakowanych plików z rozmiarami
- [ ] Ostrzeżenie, jeśli któregoś pliku brakuje

In [ ]:
# TWOJA KOLEJ: uzupełnij jedną lukę [WPISZ: …], potem uruchom.
moj_prompt = """
Kontekst:
Zakończyłeś analizę scenariusza. Wszystkie pliki wynikowe powinny być dostępne w środowisku.

Wejście:
Pliki do spakowania: `bundle.json`, `graf_postaci_wzbogacony.graphml`, `siec_wzbogacona.png`, `dialogi.csv`, `postacie.csv`, `emocje.csv`, `emocje_postaci.csv`.

Zadanie:
1. Dla każdego pliku sprawdź, czy istnieje na dysku (`os.path.exists`).
2. Spakuj istniejące pliki do archiwum ZIP o nazwie `[WPISZ: tytuł-twojego-filmu]_analiza.zip`.
3. Jeśli któregoś pliku brakuje, wypisz ostrzeżenie i kontynuuj bez niego.
4. Wyświetl link do pobrania archiwum z Colab (`google.colab.files.download`).

Pokaż wynik:
- Lista spakowanych plików z rozmiarami (KB).
- Ostrzeżenia o brakujących plikach (jeśli są).
- Link do pobrania ZIP.

Warunek poprawności:
Archiwum ZIP istnieje na dysku. Zawiera co najmniej `bundle.json` i `siec_wzbogacona.png`.

Jeśli wystąpi błąd:
Wypisz, który plik sprawia problem i jaki jest komunikat błędu.

Nie rób jeszcze:
Nie usuwaj ani nie zmieniaj oryginalnych plików.
"""
assert "[WPISZ" not in moj_prompt, \
    "⛔ Pozostała luka [WPISZ: …] — wpisz tytuł swojego filmu jako nazwę archiwum."
assert ".zip" in moj_prompt.lower(), \
    "⛔ Prompt musi zawierać nazwę archiwum z rozszerzeniem .zip."
print("✅ Prompt gotowy. Skopiuj poniższą treść do asystenta AI:\n")
print(moj_prompt)

In [ ]:
# tu wklej kod wygenerowany przez model

#### Po uruchomieniu powinieneś zobaczyć

- Link do pobrania archiwum ZIP.
- Listę plików w archiwum z rozmiarami.
- Ewentualne ostrzeżenia o brakujących plikach.

---
## Etap 5/5 — Generowanie prezentacji HTML

Ostatni krok dzieje się **poza notatnikiem** — w oknie czatu z modelem AI. To ten sam wzorzec, który znasz z generowania dashboardu SNA.

### Jak to zrobić

1. **Pobierz** archiwum ZIP z poprzedniego kroku i rozpakuj lokalnie.
2. **Otwórz** plik `prompts/prompt_dialogowy_deck.md` z repozytorium kursu.
3. **Skopiuj** cały tekst promptu (od „Jesteś ekspertem…" do końca).
4. **Otwórz** Claude (claude.ai) lub Gemini i wklej prompt.
5. **Dołącz** do wiadomości plik `bundle.json`.
6. **Wyślij** i poczekaj na wygenerowanie pliku HTML.
7. **Zapisz** odpowiedź jako `[tytuł_filmu]_prezentacja.html` i otwórz w przeglądarce.

### Co zawiera wygenerowana prezentacja

| Slajd | Zawartość |
|---|---|
| 1 | Tytuł — film, podtytuł „Głos · Płeć · Emocje" |
| 2 | Udział w dialogu × płeć (wykres słupkowy) |
| 3 | Łuk emocjonalny (wykres liniowy) |
| 4 | Emocje × płeć (wykres grupowany) |
| 5 | Sieć postaci — interaktywna (D3, kolor = płeć, rozmiar = mowa) |
| 6 | Interpretacja humanistyczna |

Estetyka i kolory są automatycznie dobierane do Twojego filmu na podstawie nazw postaci.

> **Jeśli plik HTML jest obcięty:** poproś model: *„Kontynuuj od miejsca, gdzie się zatrzymałeś"* albo wygeneruj slajdy w częściach (1–3, potem 4–6).

---
## Co dalej?

Masz teraz kompletny zestaw materiałów dla swojego filmu:

- `analiza_sieciowa_*.ipynb` + `graf_postaci.graphml` — sieć współwystąpień (moduł 1)
- `dialogi.csv` + `postacie.csv` — głos i płeć (moduł 2)
- `emocje.csv` + `emocje_postaci.csv` — emocje (moduł 3)
- `graf_postaci_wzbogacony.graphml` + `bundle.json` — synteza (moduł 4)
- `[film]_prezentacja.html` — gotowa prezentacja

**Możliwe dalsze kroki:**
- Porównaj swój film z wynikami innych studentów: czy filmy z tego samego gatunku mają podobny łuk emocjonalny?
- Dodaj wymiar czasu: podziel krawędzie na dwie warstwy (pierwsza i druga połowa) i sprawdź, jak zmienia się struktura sieci.
- Sprawdź, czy postać z najwyższym betweenness centrality to też ta, która mówi najwięcej.